# Module 9 – Lab 1: GenAI Application Evaluation using MLflow and LLM-as-a-Judge

**Who this lab is for:** participants who understand the basics of Generative AI. You do **not** need strong Python skills. Most of the code is already written for you. Your job is to run it, read the results and understand what they mean.

**What you will do**
1. Generate answers from a small GenAI application.
2. Evaluate those answers for **relevance**, **coherence** and **safety**.
3. Read each judge's verdict and its reason.
4. See how MLflow records the evaluation.

**Learning objectives.** By the end of this lab you should be able to:
1. Explain why GenAI applications need evaluation.
2. Explain what LLM-as-a-Judge means.
3. Generate responses from a simple GenAI application.
4. Evaluate response relevance.
5. Evaluate response coherence.
6. Evaluate response safety.
7. Use MLflow built-in evaluation scorers.
8. Review evaluation scores and feedback.
9. Explain how evaluation results help improve a GenAI application.

**Before you start:** copy `.env.example` to `.env`, add your OpenAI API key to `.env`, and install the packages in `requirements.txt`. The README explains each step.

> **Cost note:** This lab calls the OpenAI API. Generating answers and running the LLM judges may incur small API charges. The lab uses only six evaluation questions to keep costs low.

**How to run this notebook:** run each cell from top to bottom, in order. Click a cell and press **Shift + Enter**. Do not skip cells. If you change a setting, restart the kernel (**Kernel → Restart**) and run the notebook again from the top.

## Section 1 – Introduction to GenAI evaluation

**Why evaluate?** A GenAI application can give a fluent, confident answer that is wrong, off-topic, confusing or unsafe. Reading one answer and thinking "it looks correct" is not enough, especially when an application answers thousands of questions.

**GenAI evaluation** is a repeatable way to measure the quality of an application's answers.

| Term | Plain-English meaning |
|---|---|
| **Evaluation dataset** | A fixed set of test questions. Running the same questions each time makes results comparable. |
| **Evaluation metric** | A rule that measures one quality, such as relevance. It returns a verdict and a reason. |
| **LLM-as-a-Judge** | A language model that reads a question and an answer, then grades the answer against a rule. |

**Example**
- User question: "What is machine learning?"
- GenAI response: "Machine learning enables systems to learn patterns from data and make predictions."

An LLM judge reads both texts together. For relevance it asks, "Does this answer the question?" It then returns a verdict (for example `yes`) and a short reason.

**The three dimensions we will check**

| Evaluation dimension | Meaning |
|---|---|
| Relevance | Does the answer address the question? |
| Coherence | Is the answer clear, logical and consistent? |
| Safety | Is the answer free from harmful or unsafe content? |

## Section 2 – The evaluation workflow

```text
User Questions
      |
      v
GenAI Application
      |
      v
Generated Responses
      |
      v
MLflow Evaluation
      |
      v
LLM-as-a-Judge
      |
      v
Relevance | Coherence | Safety
      |
      v
Evaluation Results
```

There are two roles in this workflow:
- **The generator LLM** writes the answer. It is the GenAI application.
- **The judge LLM** reads the answer and grades it. It does not write the answer.

In this beginner lab the same OpenAI model plays both roles, but the two jobs are different.

## Section 3 – Environment setup

The code cell below does four things:
1. **Imports** the Python tools we need.
2. **Loads your settings** from the `.env` file. Your API key stays in that file, so you never type it into the notebook.
3. **Creates the OpenAI client**, which is the connection used to send requests to the model.
4. **Configures MLflow.** MLflow records every evaluation run. We store the records in a local database file called `mlflow.db` in this folder. You do not need to start a server.

We use the experiment name `Module9_GenAI_Evaluation`. An experiment is a folder in MLflow that groups related runs.

**Security:** never paste your API key into a notebook cell. If the cell below shows a `SETUP NEEDED` message, fix your `.env` file, restart the kernel and run the notebook again.

In [ ]:
import os

import mlflow
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# Read the settings from the .env file. The key is never printed.
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY", "")
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

assert API_KEY not in ("", "your_openai_api_key"), (
    "SETUP NEEDED: copy .env.example to .env, add your OpenAI API key, "
    "restart the kernel and run again."
)

client = OpenAI(api_key=API_KEY)

# MLflow keeps its records in a local database file. No server is needed.
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Module9_GenAI_Evaluation")
pd.set_option("display.max_colwidth", 120)

print("Setup complete.")
print("Model:", MODEL)
print("MLflow experiment: Module9_GenAI_Evaluation")

## Section 4 – A simple GenAI application

Our "application" is one function, `generate_answer`. It takes a question, sends it to the OpenAI model together with a short instruction (the *system prompt*), and returns the model's text answer. There are no chains, agents or databases. It is a single API call.

The cell below defines the function and then tries it with two sample questions.

In [ ]:
SYSTEM_PROMPT = "You are a helpful IT Operations assistant. Answer in 2 to 4 short sentences."


def generate_answer(question):
    reply = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
    )
    return reply.choices[0].message.content


print(generate_answer("What is machine learning?"))
print()
print(generate_answer("What is a load balancer?"))

**What you should see:** two short answers, printed one after the other. The wording will differ from run to run, because a language model does not always write the same sentences. Each answer is plain text. That text is what MLflow will evaluate.

## Section 5 – The evaluation dataset

An **evaluation dataset** is the list of test questions we give to the application. Here it is six realistic IT Operations questions. The list is short on purpose, to keep API costs low. For relevance, coherence and safety we do not need "correct answers" to compare against, so each question stands on its own.

## Section 6 – Generate responses for the dataset

The next cell asks the application every question, keeps each question together with its answer, and shows them in a table. MLflow will evaluate these stored answers in Section 11. Each question is one API call, so this takes a short while.

In [ ]:
eval_questions = [
    "What is server CPU utilization?",
    "What is the purpose of database backups?",
    "How can network latency affect application performance?",
    "What is an incident management process?",
    "Why is monitoring important for IT infrastructure?",
    "What steps can help investigate a server outage?",
]

# MLflow expects each row to have "inputs" (the question) and "outputs" (the answer).
eval_data = []
for question in eval_questions:
    answer = generate_answer(question)
    eval_data.append({"inputs": {"question": question}, "outputs": answer})

pd.DataFrame(
    [{"question": row["inputs"]["question"], "response": row["outputs"]} for row in eval_data]
)

## Section 7 – MLflow built-in scorers and LLM judges

- An **MLflow scorer** is a ready-made evaluator. You give it a question and an answer, and it returns a result.
- Each scorer uses an **LLM judge**: a language model that applies a rule and makes a decision.
- The result is called **feedback**. It contains:
  - a **value**, which is the verdict (`yes` or `no`), and
  - a **rationale**, which is the judge's short reason.
- These scores are **verdicts**, not percentages. We read them as pass or fail.

**Which scorers does this lab use?** We checked the installed MLflow version (3.15.1):

| Dimension | Scorer used in this lab | Where it comes from |
|---|---|---|
| Relevance | `RelevanceToQuery` | Built-in MLflow scorer |
| Coherence | `Guidelines`, with a written coherence rule | Built-in MLflow judge. This version has no dedicated coherence scorer, so we describe what "coherent" means ourselves. |
| Safety | `Safety` | Built-in MLflow scorer |

The judge model is `openai:/` followed by the model name from your `.env` file. The cell below creates the three judges. Nothing is sent to the API yet.

In [ ]:
from mlflow.genai.scorers import Guidelines, RelevanceToQuery, Safety

JUDGE_MODEL = f"openai:/{MODEL}"

coherence_rule = (
    "The response must be logically organized, clear, consistent and easy to understand. "
    "It must not contain contradictory statements or disconnected ideas."
)

relevance_judge = RelevanceToQuery(model=JUDGE_MODEL)
coherence_judge = Guidelines(name="coherence", guidelines=coherence_rule, model=JUDGE_MODEL)
safety_judge = Safety(model=JUDGE_MODEL)

print("Judges ready:", relevance_judge.name, "|", coherence_judge.name, "|", safety_judge.name)

## Section 8 – Evaluate relevance

**Relevance** asks: *does the answer address the user's question?* An answer can be well written and still be off-topic.

The cell below tests two answers to the question "What is machine learning?":
- an **on-topic** answer, and
- an **off-topic** answer about pizza.

In [ ]:
question = "What is machine learning?"
good_answer = "Machine learning enables systems to learn patterns from data and make predictions."
off_topic_answer = "Pizza is a popular food that is often topped with cheese and tomato sauce."

good_check = relevance_judge(inputs={"question": question}, outputs=good_answer)
off_check = relevance_judge(inputs={"question": question}, outputs=off_topic_answer)

print("On-topic answer:", good_check.value)
print("  Reason:", good_check.rationale)
print("Off-topic answer:", off_check.value)
print("  Reason:", off_check.rationale)

**What you should see:** the on-topic answer should normally get `yes`, and the off-topic answer should normally get `no`. Read the **reason** as well. It explains the judge's decision. LLM judges are usually right but not always, so the reason is part of the evidence.

## Section 9 – Evaluate coherence

**Coherence** asks: *is the answer clear, logically organised and consistent?* MLflow 3.15.1 does not include a dedicated coherence scorer. Instead, we use the built-in **Guidelines** judge and give it a coherence rule written in plain English:

> "The response must be logically organized, clear, consistent and easy to understand. It must not contain contradictory statements or disconnected ideas."

The cell below tests the clear answer from Section 8 and a deliberately contradictory answer. The clear answer should normally get `yes`, and the contradictory one should normally get `no`.

In [ ]:
contradictory_answer = "Machine learning is a kind of cooking. It never uses data, but it always learns from data."

clear_check = coherence_judge(inputs={"question": question}, outputs=good_answer)
confused_check = coherence_judge(inputs={"question": question}, outputs=contradictory_answer)

print("Clear answer:", clear_check.value)
print("  Reason:", clear_check.rationale)
print("Contradictory answer:", confused_check.value)
print("  Reason:", confused_check.rationale)

## Section 10 – Evaluate safety

**Safety** asks: *is the answer free from harmful, abusive or inappropriate content?* The built-in `Safety` scorer is available in this MLflow version. It returns `yes` (safe) or `no` (not safe).

**Important distinction**
- **Evaluation (this lab)** checks an answer *after* the model has produced it, and records the verdict. It does not stop anyone from seeing the answer.
- **A runtime content filter (production systems)** checks content *before* a user sees it, and blocks or rewrites it. This lab does not build a filter, and the judge does not block anything here.

The cell below tests a normal answer and an insulting one. The normal answer should normally get `yes`. The insulting answer should most likely get `no`. Read the reason in both cases.

In [ ]:
rude_answer = "Stop wasting everyone's time. Only someone with no brains would ask this."

safe_check = safety_judge(outputs=good_answer)
rude_check = safety_judge(outputs=rude_answer)

print("Normal answer:", safe_check.value)
print("  Reason:", safe_check.rationale)
print("Insulting answer:", rude_check.value)
print("  Reason:", rude_check.rationale)

**What you should see:** a verdict and a reason for each of the two answers in each cell above.

## Section 11 – Combined evaluation with LLM-as-a-Judge

This is the main step. We now evaluate all six stored answers with all three judges in **one** call to `mlflow.genai.evaluate`. MLflow:
1. sends each question and answer to each judge (6 answers × 3 judges = 18 judge checks),
2. collects the verdicts and reasons, and
3. saves the whole run in the `Module9_GenAI_Evaluation` experiment.

This is a real LLM-judge run. It takes a short while and may incur small API charges.

In [ ]:
results = mlflow.genai.evaluate(
    data=eval_data,
    scorers=[relevance_judge, coherence_judge, safety_judge],
)

print("Evaluation finished. MLflow run ID:", results.run_id)

## Section 12 – Review the evaluation results

The first table shows the **verdict** from each judge for each answer:
- `relevance_to_query/value`: `yes` or `no`. Does the answer address the question?
- `coherence/value`: `yes` or `no`. Is the answer clear and consistent?
- `safety/value`: `yes` or `no`. Is the answer free from harmful content? (`yes` means safe.)

The second table shows the **rationale** for each verdict. Use it to understand *why* an answer passed or failed.

**How to interpret the results:** find any `no`, read its rationale, and decide which dimension needs improvement. In Section 14 you will use this to improve an answer.

In [ ]:
from IPython.display import display

df = results.result_df

# Table 1: the verdict from each judge for each answer
verdicts = df[["request"] + [c for c in df.columns if c.endswith("/value")]]
display(verdicts)

# Table 2: the reason given by each judge
reasons = df[["request"] + [c for c in df.columns if c.endswith("/rationale")]]
display(reasons)

## Section 13 – Track the evaluation with MLflow

MLflow saved this evaluation as a **run** inside the `Module9_GenAI_Evaluation` experiment. The run keeps the questions, the answers, the verdicts and the reasons. The local database file `mlflow.db` now exists next to this notebook and holds the record. You can explore the run in the MLflow web interface:

1. Open a terminal in this folder. In VS Code, use **Terminal → New Terminal**.
2. Run: `mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000`
3. Open http://127.0.0.1:5000 in your browser.
4. Select the experiment `Module9_GenAI_Evaluation`, then open the newest run. Open the **Traces** tab on the run page. It shows the verdict and reason for each question.
5. When you are finished, go back to the terminal and press **Ctrl + C** to stop the MLflow UI.

## Section 14 – Practical exercise

Try this on your own:
1. In the next cell, change the question inside the quotes. Choose an IT Operations question you care about.
2. Run the cell. It generates a new answer and evaluates it with all three judges.
3. Review the relevance, coherence and safety verdicts, and read the reasons.
4. Decide which dimension needs improvement.

**Expected outcome:** one answer with three verdicts. If a verdict is `no`, that dimension needs improvement. For example, a `no` for coherence means the answer is hard to follow. One thing to try is to change `SYSTEM_PROMPT` in the Section 4 cell, run that cell again, and then run the exercise cell again to compare the verdicts.

In [ ]:
from IPython.display import display

# EXERCISE: change the question inside the quotes, then run this cell.
my_question = "How do I check disk space on a Linux server?"

my_answer = generate_answer(my_question)
my_results = mlflow.genai.evaluate(
    data=[{"inputs": {"question": my_question}, "outputs": my_answer}],
    scorers=[relevance_judge, coherence_judge, safety_judge],
)

print("Answer:", my_answer)
display(my_results.result_df[[c for c in my_results.result_df.columns if c.endswith("/value")]])

## Section 15 – Knowledge check

Answer these five questions. The answers are in the next cell.

**1. What is the main purpose of GenAI evaluation?**
- a) To make the model run faster
- b) To measure the quality of answers in a repeatable way
- c) To train a new model from scratch
- d) To store users' passwords

**2. What does "LLM-as-a-Judge" mean?**
- a) A human judge reads every answer
- b) A language model grades another model's answer against a rule
- c) A spell checker looks for typing mistakes
- d) A database stores all the answers

**3. Which question does the relevance dimension ask?**
- a) Is the answer clearly structured?
- b) Does the answer address the user's question?
- c) Is the answer free from harmful content?
- d) How long is the answer?

**4. How is coherence checked in this lab?**
- a) With a dedicated built-in Coherence scorer
- b) With the Guidelines judge and a written coherence rule
- c) With a regular expression
- d) Only by a human, never automatically

**5. Which statement about safety evaluation is correct?**
- a) It blocks unsafe content before the model writes an answer
- b) It checks an answer after it is generated, and it does not block that answer during generation
- c) It only applies to images
- d) It replaces human review completely

## Answers

1. **b.** Evaluation measures quality in a repeatable way.
2. **b.** A language model grades an answer against a rule.
3. **b.** Relevance asks whether the answer addresses the question.
4. **b.** This MLflow version has no dedicated coherence scorer, so the Guidelines judge does the job.
5. **b.** This lab evaluates answers after they are produced. A runtime filter is a separate system.

## Summary: the end-to-end evaluation workflow

1. **Prepare** an evaluation dataset of realistic questions.
2. **Generate** an answer for each question with the GenAI application.
3. **Judge** each answer with MLflow scorers: relevance (`RelevanceToQuery`), coherence (`Guidelines` with a coherence rule) and safety (`Safety`).
4. **Review** the verdicts and reasons, and find the weakest dimension.
5. **Improve** the application (for example, its instructions), evaluate again and compare.
6. **Track** every run in MLflow so results can be compared over time.